FYP YOLOv11 Medical Prescription Detection Model

Optimized for medical prescription text field detection with YOLOv11-Large

```
# This is formatted as code
```



In [3]:
# ==========================
# STEP 1 – Setup Environment
!pip install ultralytics roboflow --quiet --upgrade
!pip install roboflow

In [4]:
# Verify YOLOv11 is available
# from ultralytics import YOLO
# print("✅ Ultralytics version:", YOLO.verion)

import ultralytics
print(ultralytics.__version__)

8.3.241


In [5]:
!pip install torch


In [6]:
import torch
print(torch.__version__)  # Check the installed version of PyTorch
print(torch.cuda.is_available())  # Check if CUDA (GPU support) is available


2.7.1+cu118
True


In [7]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA RTX A2000 12GB


In [8]:
# ============================================
# STEP 1b – Local Save Directory and Robust GPU / Device Detection
# ============================================
import os
import torch

# Local save directory (adjust path if needed)
save_dir = r"$PWD"
os.makedirs(save_dir, exist_ok=True)

# GPU device selection - robust flags
cuda_available = torch.cuda.is_available()
cuda_count = torch.cuda.device_count() if cuda_available else 0

# Default device selection — prefer first visible GPU 'cuda:0' (or change if needed)
if cuda_available and cuda_count > 0:
    # You can change this to another index if you have multiple GPUs
    cuda_index = 0
    device = f"cuda:{cuda_index}"
    device_arg = device  # for Ultralytics, 'cuda:0' or int 0 are both fine
else:
    device = 'cpu'
    device_arg = 'cpu'

print("✅ Local save directory created at:", save_dir)
print(f"🔌 CUDA available: {cuda_available}, CUDA count: {cuda_count}")
if cuda_available and cuda_count > 0:
    print(f"🔌 Using CUDA device: {torch.cuda.get_device_name(cuda_index)} (index {cuda_index})")
print(f"🔌 Using final device string: {device} (device_arg: {device_arg})")

# Optional: explicitly set CUDA_VISIBLE_DEVICES if you want to restrict GPUs (uncomment to use)
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Limits visible GPUs to GPU 0

# QUICK TEST: Verify save_dir is writable
try:
    test_file = os.path.join(save_dir, 'test_write.txt')
    with open(test_file, 'w', encoding='utf-8') as fh:
        fh.write('Local save_dir test — this confirms we can write files here.')
    print(f"✅ Wrote test file to: {test_file}")
except Exception as e:
    print(f"⚠️ Failed to write test file: {e}")


✅ Local save directory created at: $PWD
🔌 CUDA available: True, CUDA count: 1
🔌 Using CUDA device: NVIDIA RTX A2000 12GB (index 0)
🔌 Using final device string: cuda:0 (device_arg: cuda:0)
✅ Wrote test file to: $PWD/test_write.txt


In [9]:
# QUICK TEST: Verify save_dir is writable

test_file = os.path.join(save_dir, 'test_write.txt')
with open(test_file, 'w', encoding='utf-8') as fh:
    fh.write('Local save_dir test — this confirms we can write files here.')

print(f"✅ Wrote test file to: {test_file}")


✅ Wrote test file to: $PWD/test_write.txt


# NOTE: Local GPU / No Google Drive

This notebook has been adapted to run locally on your machine and save all files to a local folder instead of Google Drive.

- Save path (local): `save_dir` variable (default set to `D:\giki books\FYP\yolo_model\FYP_Prescription_YOLOv11`)
- GPU selection: `device` / `device_arg` -> uses GPU if available, else CPU
- Access: Use rustdesk or any remote desktop — the notebook will still run on the local machine and use local GPU resources

If you want to change where outputs are saved, modify `save_dir` accordingly.


In [10]:
# ============================================
# STEP 2 – Download Roboflow Dataset
# ============================================
from roboflow import Roboflow

rf = Roboflow(api_key="HZ4iOiyleovq0nCt9ntz")
project = rf.workspace("personal-yyejy").project("fyp-fqsqv")
version = project.version(1)

# Download in YOLOv11 format (Roboflow supports this now)
dataset = version.download("yolov11")

print(f"✅ Dataset downloaded to: {dataset.location}")

loading Roboflow workspace...
loading Roboflow project...
✅ Dataset downloaded to: /home/ocrdetection/Downloads/fyp/FYP-1


In [11]:
# ============================================
# STEP 2b – Move Roboflow dataset into local save_dir
# ============================================
import shutil
import os

# Ensure dataset exists and copy into save_dir/dataset if not already there
dataset_target = os.path.join(save_dir, 'dataset')
if hasattr(dataset, 'location') and os.path.exists(dataset.location):
    if os.path.abspath(dataset.location) != os.path.abspath(dataset_target):
        print(f"Copying dataset from {dataset.location} to {dataset_target}...")
        shutil.copytree(dataset.location, dataset_target, dirs_exist_ok=True)
        print("✅ Dataset copied locally")
    else:
        print("✅ Dataset already in local save directory")

# Update dataset.location to point at our local copy
if os.path.exists(dataset_target):
    dataset.location = dataset_target

print(f"🔹 Dataset location set to: {dataset.location}")


Copying dataset from /home/ocrdetection/Downloads/fyp/FYP-1 to $PWD/dataset...
✅ Dataset copied locally
🔹 Dataset location set to: $PWD/dataset


In [12]:
# ============================================
# STEP 3 – Initialize YOLOv11 Model (robust import & device verification)
# ============================================
try:
    from ultralytics import YOLO
    ultralytics_available = True
except Exception as e:
    print(f"⚠️ Ultralytics not installed in this environment: {e}")
    ultralytics_available = False

if ultralytics_available:
    # Create/load the model and check where it is located
    model = YOLO("yolo11l.pt")  # Using Large variant
    print("✅ YOLOv11-Large model loaded")

    # VERIFY: where is the underlying PyTorch model/device located?
    try:
        underlying_device = None
        # Try common attributes for ultralytics versions
        for candidate in ('device', 'model', 'pt'):
            if candidate == 'device' and hasattr(model, 'device'):
                underlying_device = model.device
                break
            if candidate == 'model' and hasattr(model, 'model') and hasattr(model.model, 'device'):
                underlying_device = model.model.device
                break
            if candidate == 'pt' and hasattr(model, 'pt') and hasattr(model.pt, 'device'):
                underlying_device = model.pt.device
                break

        if underlying_device is not None:
            try:
                import torch as _torch
                # Convert to string for readability if torch.device
                if isinstance(underlying_device, _torch.device):
                    underlying_device_str = str(underlying_device)
                else:
                    underlying_device_str = repr(underlying_device)
                print(f"🔎 Model reported device attribute: {underlying_device_str}")
            except Exception:
                print(f"🔎 Model device attribute: {underlying_device}")
        else:
            print("🔎 Model does not have an easily found 'device' attribute. We'll move it explicitly below if needed.")
    except Exception as e:
        print(f"⚠️ Could not read model device attribute: {e}")

    # Move the model to the detected device if needed
    try:
        if device.startswith('cuda'):
            print(f"🔧 Moving model to {device}")
            # Some Ultralytics versions support model.to(); we try that first
            try:
                model.to(device)
            except Exception:
                # As a robust fallback, move the underlying PyTorch model
                if hasattr(model, 'model') and hasattr(model.model, 'to'):
                    model.model.to(device)
        else:
            # Ensure the model is on CPU
            try:
                model.to('cpu')
            except Exception:
                if hasattr(model, 'model') and hasattr(model.model, 'to'):
                    model.model.to('cpu')
    except Exception as e:
        print(f"⚠️ Failed to move model to {device}: {e}")

    # Re-check where the model is located for final verification
    try:
        final_model_device = None
        if hasattr(model, 'device'):
            final_model_device = model.device
        elif hasattr(model, 'model') and hasattr(model.model, 'device'):
            final_model_device = model.model.device

        if final_model_device is not None:
            print(f"✅ Final model device: {final_model_device}")
        else:
            print("⚠️ Final model device could not be determined automatically")
    except Exception as e:
        print(f"⚠️ Error determining final model device: {e}")
else:
    print("⚠️ Skipping YOLO model creation as ultralytics is not installed in this kernel. Install the package to load models.")


✅ YOLOv11-Large model loaded
🔎 Model reported device attribute: cpu
🔧 Moving model to cuda:0
✅ Final model device: cuda:0


In [13]:
# Helper: Check what device the Ultralytics YOLO model is using

def check_yolo_model_device(model, expected_device=None):
    """
    Attempts to detect the device returned by the YOLO object.
    Returns a tuple (bool_is_expected, actual_device_str)
    """
    import torch

    actual_device = None
    # Check common attributes
    if hasattr(model, 'device'):
        actual_device = getattr(model, 'device')
    elif hasattr(model, 'model') and hasattr(model.model, 'device'):
        actual_device = model.model.device
    elif hasattr(model, 'pt') and hasattr(model.pt, 'device'):
        actual_device = model.pt.device
    else:
        # As a fallback, inspect the parameters of the underlying PyTorch model
        try:
            if hasattr(model, 'model') and hasattr(model.model, 'parameters'):
                params = list(model.model.parameters())
                if len(params) > 0:
                    actual_device = params[0].device
            elif hasattr(model, 'parameters'):
                params = list(model.parameters())
                if len(params) > 0:
                    actual_device = params[0].device
        except Exception:
            actual_device = None

    if isinstance(actual_device, torch.device):
        actual_device_str = str(actual_device)
    else:
        actual_device_str = repr(actual_device)

    is_expected = None
    if expected_device is None:
        # If no expected device passed, just return the device string and a True/False for CUDA availability
        is_expected = True if (('cuda' in actual_device_str) == torch.cuda.is_available()) else False
    else:
        # Compare normalized strings; expected_device may be 'cuda:0' or 'cuda'
        try:
            if expected_device.startswith('cuda') and 'cuda' in actual_device_str:
                is_expected = True
            elif expected_device == 'cpu' and 'cpu' in actual_device_str:
                is_expected = True
            else:
                is_expected = False
        except Exception:
            is_expected = False

    return is_expected, actual_device_str

# Call it (will print info only if model created)
try:
    if "model" in globals() and model is not None:
        result_ok, actual_dev = check_yolo_model_device(model, expected_device=device)
        print(f"🔎 check_yolo_model_device -> expected: {device}, actual: {actual_dev}, ok: {result_ok}")
except Exception:
    pass


🔎 check_yolo_model_device -> expected: cuda:0, actual: cuda:0, ok: True


In [14]:
import sys
print(sys.executable)
print(sys.path)
import matplotlib
print(matplotlib.__file__)

/home/ocrdetection/yoloenv/bin/python
['/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/home/ocrdetection/yoloenv/lib/python3.10/site-packages', '/tmp/tmpra58zq5b']
/home/ocrdetection/yoloenv/lib/python3.10/site-packages/matplotlib/__init__.py


In [ ]:
# ============================================
# STEP 4 – Train YOLOv11 Model (OPTIMIZED)
# ============================================

# Validate the model device before training
try:
    ok, actual_dev = check_yolo_model_device(model, expected_device=device)
    if not ok:
        print(f"⚠️ Warning: model device ({actual_dev}) does not match expected device ({device}).")
        print("If you intended to run on GPU, ensure CUDA is available and model is moved to the correct device")
    else:
        print(f"✅ Model device verified: {actual_dev}")
except Exception:
    print("🔎 Could not verify model device automatically. Proceeding with the given 'device' parameter for training.")

# Training configuration optimized for medical prescriptions
results = model.train(
    # Dataset
    data=f"{dataset.location}/data.yaml",

    # Training parameters
    epochs=150,              # Increased for better convergence
    imgsz=1024,             # High resolution for small text
    batch=4,                # Adjust based on your GPU (reduce if OOM)

    # Optimizer settings
    optimizer='AdamW',      # Better than SGD for small objects
    lr0=0.001,             # Initial learning rate
    lrf=0.01,              # Final learning rate (lr0 * lrf)
    momentum=0.937,
    weight_decay=0.0005,

    # Augmentation (reduced for text preservation)
    mosaic=0.5,            # 50% mosaic (less aggressive than default)
    mixup=0.0,             # Disabled - avoid blending text
    copy_paste=0.0,        # Disabled - preserve text integrity
    degrees=5.0,           # Minimal rotation (prescriptions usually upright)
    translate=0.1,         # Small translation
    scale=0.3,             # Less aggressive scaling for small text
    shear=2.0,             # Minimal shear
    perspective=0.0,       # No perspective transform
    flipud=0.0,            # No vertical flip (text orientation matters)
    fliplr=0.0,            # No horizontal flip (preserves reading direction)

    # Training settings
    patience=30,           # Early stopping patience
    save=True,             # Save checkpoints
    save_period=10,        # Save every 10 epochs
    cache=False,           # Set True if enough RAM (speeds up training)
    device=device_arg,     # GPU device or 'cpu'
    workers=4,             # Data loading workers (reduce if CPU bottleneck)

    # Output settings
    project=save_dir,
    name="yolov11l_prescription_v1",
    exist_ok=True,

    # Advanced settings
    rect=True,             # Rectangular training (preserves aspect ratio)
    cos_lr=True,           # Cosine learning rate scheduler
    close_mosaic=10,       # Disable mosaic in last 10 epochs
    amp=True,              # Automatic Mixed Precision (faster training)

    # Validation
    val=True,              # Validate during training
    plots=True,            # Generate plots
    save_json=True,        # Save results as JSON
)

print("\n✅ Training completed!")
print(f"📊 Best model saved to: {save_dir}/yolov11l_prescription_v1/weights/best.pt")


✅ Model device verified: cuda:0
Ultralytics 8.3.241 🚀 Python-3.10.12 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A2000 12GB, 11896MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=$PWD/dataset/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=0.5, multi_scale=False, name=yolov11l_prescription_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask

OutOfMemoryError: CUDA out of memory. Tried to allocate 58.00 MiB. GPU 0 has a total capacity of 11.62 GiB of which 25.19 MiB is free. Process 1456597 has 136.32 MiB memory in use. Including non-PyTorch memory, this process has 10.81 GiB memory in use. Of the allocated memory 10.18 GiB is allocated by PyTorch, and 440.12 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ============================================
# STEP 5 – View Training Results
# ============================================
from IPython.display import Image, display

print("\n📈 Training Results:")
display(Image(filename=f"{save_dir}/yolov11l_prescription_v1/results.png"))

print("\n📊 Confusion Matrix:")
display(Image(filename=f"{save_dir}/yolov11l_prescription_v1/confusion_matrix.png"))

In [ ]:
# ============================================
# STEP 6 – Load Best Model for Evaluation
# ============================================
best_model_path = f"{save_dir}/yolov11l_prescription_v1/weights/best.pt"
model = YOLO(best_model_path)
print(f"✅ Loaded best model from: {best_model_path}")

In [2]:

# ============================================
# STEP 7 – Evaluate on Test Set
# ============================================
data_yaml = f"{dataset.location}/data.yaml"

print("\n🔹 Evaluating model on TEST split...\n")
test_results = model.val(
    data=data_yaml,
    split="test",
    imgsz=1024,
    conf=0.25,      # Lower confidence threshold for detection
    iou=0.4,        # IoU threshold for NMS
    max_det=50,     # Max detections per image (adjust for your prescription fields)
    plots=True
)

print("\n✅ TEST SET METRICS")
print("=" * 50)
print(f"mAP@50:     {test_results.box.map50:.4f}")
print(f"mAP@50-95:  {test_results.box.map:.4f}")
print(f"Precision:  {test_results.box.mp:.4f}")
print(f"Recall:     {test_results.box.mr:.4f}")
print(f"F1-Score:   {(2 * test_results.box.mp * test_results.box.mr) / (test_results.box.mp + test_results.box.mr):.4f}")
print("=" * 50)

NameError: name 'dataset' is not defined

In [ ]:

# ============================================
# STEP 8 – Evaluate on Validation Set
# ============================================
print("\n🔹 Evaluating model on VALIDATION split...\n")
val_results = model.val(
    data=data_yaml,
    split="val",
    imgsz=1024,
    conf=0.25,
    iou=0.4,
    max_det=50,
    plots=True
)

print("\n✅ VALIDATION SET METRICS")
print("=" * 50)
print(f"mAP@50:     {val_results.box.map50:.4f}")
print(f"mAP@50-95:  {val_results.box.map:.4f}")
print(f"Precision:  {val_results.box.mp:.4f}")
print(f"Recall:     {val_results.box.mr:.4f}")
print(f"F1-Score:   {(2 * val_results.box.mp * val_results.box.mr) / (val_results.box.mp + val_results.box.mr):.4f}")
print("=" * 50)

In [ ]:
# ============================================
# STEP 9 – Per-Class Performance Analysis
# ============================================
print("\n📊 Per-Class Performance (Test Set):")
print("=" * 70)
print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'mAP@50':<12}")
print("=" * 70)

# Get class names from data.yaml
import yaml
with open(data_yaml, 'r') as f:
    data_config = yaml.safe_load(f)
    class_names = data_config['names']

# Display per-class metrics
for i, class_name in enumerate(class_names):
    if i < len(test_results.box.p):  # Check if class exists in results
        precision = test_results.box.p[i] if hasattr(test_results.box, 'p') else 0
        recall = test_results.box.r[i] if hasattr(test_results.box, 'r') else 0
        map50 = test_results.box.map50_per_class[i] if hasattr(test_results.box, 'map50_per_class') else 0
        print(f"{class_name:<20} {precision:<12.4f} {recall:<12.4f} {map50:<12.4f}")
print("=" * 70)

In [ ]:
# ============================================
# STEP 10 – Run Predictions on Test Images
# ============================================
import glob

print("\n🔹 Running predictions on sample test images...")
# Ensure dataset loc is current; dataset.location should already be updated by the dataset cell
test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")[:10]
pred_dir = os.path.join(save_dir, "predictions_test")
os.makedirs(pred_dir, exist_ok=True)

for img_path in test_images:
    results = model.predict(
        source=img_path,
        conf=0.25,           # Detection confidence threshold
        iou=0.4,             # NMS IoU threshold
        imgsz=1024,
        save=True,
        save_txt=True,       # Save labels as TXT files
        save_conf=True,      # Include confidence in labels
        project=pred_dir,
        name="test_outputs",
        exist_ok=True,
        line_width=2,
        show_labels=True,
        show_conf=True,
        device=device_arg,   # Force Ultralytics to use selected device
    )
    print(f"✓ Predicted: {os.path.basename(img_path)}")


In [ ]:
# ============================================
# STEP 11 – Display Sample Predictions
# ============================================
print("\n✅ Showing sample prediction images...")
pred_folder = f"{pred_dir}/test_outputs"
sample_preds = glob.glob(f"{pred_folder}/*.jpg")[:5]

for img_path in sample_preds:
    print(f"\n📸 {os.path.basename(img_path)}")
    display(Image(filename=img_path, width=800))

In [ ]:
# ============================================
# STEP 12 – Export Model for Deployment
# ============================================
print("\n🔹 Exporting model for deployment...")

# Export to ONNX (recommended for production)
model.export(format="onnx", imgsz=1024, simplify=True)
print("✅ Model exported to ONNX format")
# Optional: Export to other formats
# model.export(format="torchscript")  # TorchScript
# model.export(format="tflite")       # TensorFlow Lite
# model.export(format="openvino")     # OpenVINO

In [ ]:
# ============================================
# STEP 13 – Save Model Backup Locally (safe)
# ============================================
import shutil
import os

# Create a local backup folder inside save_dir
backup_dir = os.path.join(save_dir, "backup")
os.makedirs(backup_dir, exist_ok=True)

# Copy best weights to backup if it exists
best_weights = os.path.join(save_dir, "yolov11l_prescription_v1", "weights", "best.pt")
backup_path = os.path.join(backup_dir, "best_model_backup.pt")
if os.path.exists(best_weights):
    shutil.copy(best_weights, backup_path)
    print(f"✅ Best model backed up locally to: {backup_path}")
else:
    print(f"⚠️ Best weights not found at {best_weights} — skipping backup (run training first)")


In [ ]:
# ============================================
# STEP 14 – Generate Model Performance Report
# ============================================
print("\n" + "="*70)
print("🎯 FINAL MODEL PERFORMANCE SUMMARY")
print("="*70)
print(f"\n📁 Model Location: {best_model_path}")
print(f"📊 Training Epochs: 150")
print(f"🖼️  Image Size: 1024x1024")
print(f"⚙️  Model: YOLOv11-Large")
print(f"\n📈 Test Set Performance:")
print(f"   • mAP@50:     {test_results.box.map50:.2%}")
print(f"   • mAP@50-95:  {test_results.box.map:.2%}")
print(f"   • Precision:  {test_results.box.mp:.2%}")
print(f"   • Recall:     {test_results.box.mr:.2%}")
print(f"\n💾 Saved Outputs:")
print(f"   • Best weights: {best_weights}")
print(f"   • Training plots: {save_dir}/yolov11l_prescription_v1/results.png")
print(f"   • Predictions: {pred_folder}")
print("="*70)

In [ ]:
# ============================================
# STEP 15 – Create Local ZIP of Model (Optional)
# ============================================
print("\n🔹 Creating local ZIP file of the trained model...")
import shutil
import os

model_folder = os.path.join(save_dir, "yolov11l_prescription_v1")
zip_basename = os.path.join(os.getcwd(), "yolov11_trained_model")
zip_path = shutil.make_archive(zip_basename, 'zip', model_folder)

print(f"✅ Local ZIP created: {zip_path}")
print("\n🚀 YOLOv11 training pipeline completed successfully!")


In [ ]:
# ============================================
# Inference Function for OCR Pipeline (updated with device control)
# ============================================

def detect_prescription_fields(image_path, model_path, conf_threshold=0.25, device_arg=None):
    """
    Detect prescription fields from an image

    Args:
        image_path: Path to prescription image
        model_path: Path to trained YOLOv11 model
        conf_threshold: Confidence threshold for detections
        device_arg: 'cpu' or 'cuda:0' etc. If None, will use CPU or available GPU.

    Returns:
        List of detected boxes with class names and coordinates
    """
    # Lazy import to avoid overhead unless called
    from ultralytics import YOLO
    import torch

    # If device arg is None, infer from torch availability
    if device_arg is None:
        device_arg = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    model = YOLO(model_path)

    # Try to move the model to the target device
    try:
        if device_arg.startswith('cuda'):
            model.to(device_arg)
        else:
            model.to('cpu')
    except Exception:
        # If using an older or different Ultralytics API, try direct manipulation
        try:
            if hasattr(model, 'model') and hasattr(model.model, 'to'):
                model.model.to(torch.device(device_arg))
        except Exception:
            pass

    results = model.predict(source=image_path, conf=conf_threshold, imgsz=1024, device=device_arg)

    detections = []
    for result in results:
        boxes = result.boxes
        for box in boxes:
            detection = {
                'class': result.names[int(box.cls[0])],
                'confidence': float(box.conf[0]),
                'bbox': box.xyxy[0].tolist(),  # [x1, y1, x2, y2]
                'bbox_normalized': box.xywhn[0].tolist()  # [x_center, y_center, width, height]
            }
            detections.append(detection)

    return detections

# Example usage:
# detections = detect_prescription_fields("prescription.jpg", best_model_path, device_arg=device_arg)
# for det in detections:
#     print(f"{det['class']}: {det['confidence']:.2f} at {det['bbox']}")

print("\n✅ Inference function 'detect_prescription_fields()' defined and ready to use!")
